# 02 · Core simulation mapping: OrchidShield triage agent

This notebook covers Phase 1, Step 2. It builds a **synthetic** SOC environment, stabilises the agent's outputs
with fixed seeds, and benchmarks the algorithm families the agent uses. Each section states a **hypothesis**,
measures it, and notes the **failure cases** found. All numbers can be reproduced: run top to bottom and the
same seed gives the same results, which are written to `data/results/simulation_metrics.json`.

In [1]:
import sys, json, time, random
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "configs").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from core.config import load_settings
S = load_settings(); SEED = S.app.seed
random.seed(SEED); np.random.seed(SEED)
RESULTS = {"seed": SEED, **S.run_metadata()}
print(S.run_metadata())

{'app_version': '1.0.0', 'config_version': '2026.11.16-1', 'config_sha': '5937ab1d8b2d', 'seed': 1337, 'python': '3.13.18'}


## 1. Environment and PEAS recap
Assets, users and indicators are generated deterministically. The environment is **partially observable**
(the true compromise state is hidden and some hosts have no EDR), **stochastic**, **sequential**, **dynamic**
(the attacker moves), **discrete** at the decision level, and **multi-agent** (attacker vs. defender).

In [2]:
from sim.generator import make_environment
env = make_environment(SEED)
A = env["assets"]
pd.DataFrame(A.values())[["asset_id","asset_type","criticality","zone","edr_installed","in_maintenance"]]

,asset_id,asset_type,criticality,zone,edr_installed,in_maintenance
0,fw-01,firewall,4,dmz,True,False
1,web-01,server,3,dmz,True,False
2,web-02,server,3,dmz,True,False
3,srv-app-01,server,4,prod,True,False
4,srv-app-02,server,3,prod,True,True
5,srv-app-03,server,3,prod,True,False
6,srv-db-01,database,5,prod,True,False
7,srv-db-02,database,5,prod,True,False
8,dc-01,domain_controller,5,mgmt,True,False
9,siem-01,siem,4,mgmt,True,False


## 2. Problem formulation and uninformed vs. informed search
**State:** the host the attacker controls. **Initial state:** a compromised workstation. **Goal:** any crown jewel
(criticality ≥ 4). **Actions:** traverse a network edge. **Cost:** exploitation difficulty (crossing zones costs more).

*Hypothesis H1:* A* with the hop heuristic finds the same optimal cost as UCS but expands fewer nodes. BFS and DFS
are not cost-optimal.

In [3]:
from reasoning import search
G = search.build_graph(A)
crown = {a for a,v in A.items() if v["criticality"] >= 4}
rows = []
for start in ["ws-001","ws-003","ws-006","ws-008","ws-011","web-01","file-01"]:
    goal = lambda n, s=start: n in crown and n != s
    h = search.hop_heuristic(G, crown - {start})
    for res in [search.bfs(G,start,goal), search.dfs(G,start,goal), search.iddfs(G,start,goal),
                search.ucs(G,start,goal), search.greedy(G,start,goal,h), search.astar(G,start,goal,h)]:
        rows.append({"start": start, "algo": res.algorithm, "cost": res.cost, "expanded": res.expanded,
                     "path": " → ".join(res.path)})
df = pd.DataFrame(rows)
summary = df.groupby("algo")[["cost","expanded"]].mean().round(2)
opt = df[df.algo=="UCS"].set_index("start").cost
summary["optimal_rate"] = df.groupby("algo").apply(lambda g: float((g.set_index("start").cost <= opt + 1e-9).mean()))
RESULTS["search"] = summary.reset_index().to_dict(orient="records")
summary

,cost,expanded,optimal_rate
algo,,,
A*,2.29,5.00,1.000000
BFS,3.21,1.43,0.285714
DFS,3.71,2.57,0.142857
Greedy,3.21,1.43,0.285714
IDDFS,3.21,4.43,0.285714
UCS,2.29,10.43,1.000000


In [4]:
# heuristic admissibility / consistency check over every node
h = search.hop_heuristic(G, crown)
adm = all(h(n) <= min((search.dijkstra_all(G,n).get(g, 1e9) for g in crown), default=1e9) + 1e-9 for n in G)
con = all(h(n) <= c + h(m) + 1e-9 for n in G for m, c in G[n])
RESULTS["heuristic"] = {"admissible": adm, "consistent": con}
print("admissible:", adm, " consistent:", con)

admissible: True  consistent: True


## 3. Optimisation: which hosts to isolate (local search)
Choose at most *k* hosts to isolate so that the criticality reachable from a compromised host is minimised, plus a
disruption penalty. Domain controllers and the SIEM are forbidden.

*Hypothesis H2:* hill climbing with restarts and simulated annealing reach the exhaustive optimum on this graph
with far fewer evaluations.

In [5]:
rows = []
for comp in ["ws-004","ws-008","web-01"]:
    p = search.SegmentationProblem(A, {comp}, k=2, forbidden={"dc-01","siem-01",comp})
    t=time.perf_counter(); ex=search.exhaustive(p); te=time.perf_counter()-t
    hc=search.hill_climb(p, random.Random(SEED)); sa=search.simulated_annealing(p, random.Random(SEED))
    rows.append({"compromised": comp, "optimum": ex[1], "opt_set": sorted(ex[0]), "exhaustive_evals": ex[2],
                 "hill_climb": hc[1], "hc_evals": hc[2], "annealing": sa[1], "sa_evals": sa[2]})
opt_df = pd.DataFrame(rows); RESULTS["local_search"] = rows; opt_df

,compromised,optimum,opt_set,exhaustive_evals,hill_climb,hc_evals,annealing,sa_evals
0,ws-004,3.5,"[file-01, jump-01]",254,3.5,3170,3.5,1500
1,ws-008,3.5,"[file-01, jump-01]",254,3.5,3170,3.5,1500
2,web-01,4.0,"[fw-01, srv-app-01]",254,4.0,3674,4.0,1500


**Failure case found:** the landscape is full of plateaus, because isolating a single host rarely reduces reach until
the cut is complete. With 5 restarts, hill climbing scored 21.5 against an optimum of 3.5, and simulated annealing
with 400 steps did the same. Raising the defaults to 20 restarts and 1,500 steps fixed both (see `reasoning/search.py`).

## 4. Adversarial search: defender vs. attacker
MAX (defender) isolates one host per turn, and MIN (attacker) spreads to an adjacent host. The utility is the
negative squared criticality of the hosts lost, minus the cost of isolation.

*Hypothesis H3:* alpha-beta returns the same value as minimax with fewer nodes. Expectimax (with a random attacker)
is never worse for the defender than minimax.

In [6]:
from reasoning.adversarial import ContainmentGame
rows=[]
for comp in [{"ws-004"},{"dc-01"},{"srv-app-03"}]:
    for depth in (2,3,4):
        r={"compromised":",".join(comp),"depth":depth}
        for m in ("minimax","alphabeta","expectimax"):
            g=ContainmentGame(A,{"dc-01","siem-01"}); t=time.perf_counter()
            mv,val,nodes=g.best_isolation(comp,depth,m)
            r[f"{m}_move"]=mv; r[f"{m}_value"]=round(val,2); r[f"{m}_nodes"]=nodes
        rows.append(r)
adv=pd.DataFrame(rows); RESULTS["adversarial"]=rows
adv[["compromised","depth","minimax_value","alphabeta_value","minimax_nodes","alphabeta_nodes","alphabeta_move","expectimax_value"]]

,compromised,depth,minimax_value,alphabeta_value,minimax_nodes,alphabeta_nodes,alphabeta_move,expectimax_value
0,ws-004,2,-0.6,-0.6,9,9,ws-004,-0.60
1,ws-004,3,-0.6,-0.6,58,44,ws-004,-0.60
2,ws-004,4,-0.6,-0.6,547,126,ws-004,-0.60
3,dc-01,2,-41.0,-41.0,37,17,NaN,-35.83
4,dc-01,3,-41.0,-41.0,293,107,NaN,-29.17
5,dc-01,4,-54.8,-54.8,2436,317,jump-01,-40.00
6,srv-app-03,2,-1.8,-1.8,23,15,srv-app-03,-1.80
7,srv-app-03,3,-1.8,-1.8,180,59,srv-app-03,-1.80
8,srv-app-03,4,-1.8,-1.8,1521,177,srv-app-03,-1.80


## 5. CSP: scheduling response tasks
Variables are response tasks. Domains are (analyst, slot) pairs. The constraints are skills, seniority for
approval-gated work, one task per analyst per slot, precedence, and separation of duties.

*Hypothesis H4:* MRV+LCV with forward checking or MAC needs far fewer assignments than plain backtracking, and
min-conflicts solves it quickly.

In [7]:
from reasoning import csp
analysts=[a.model_dump() for a in S.csp.analysts]
def problem(n_inc, n_slots):
    T=[]
    for i in range(n_inc):
        T += [csp.Task(f"triage{i}","triage"), csp.Task(f"enrich{i}","endpoint"),
              csp.Task(f"isolate{i}","endpoint",True,after=[f"enrich{i}"],separate_from=[f"triage{i}"]),
              csp.Task(f"disable{i}","identity",True,after=[f"triage{i}"])]
    # dependants listed first = a deliberately poor static variable order for plain backtracking
    return csp.build_schedule_csp(T[::-1], analysts, list(range(n_slots)))
rows=[]
for n, slots in [(2,3),(3,5),(4,6),(3,4)]:      # last instance is over-constrained (unsatisfiable)
    for name,kw in [("backtracking",{}),("MRV+LCV",{"mrv":True,"lcv":True}),("MRV+FC",{"mrv":True,"forward_check":True}),("MAC(AC-3)",{"mrv":True,"lcv":True,"mac":True})]:
        c=problem(n,slots); t=time.perf_counter(); sol=csp.backtrack(c,**kw)
        rows.append({"incidents":n,"slots":slots,"solver":name,"solved":bool(sol and csp.verify(c,sol)),"assignments":c.stats["assignments"],"backtracks":c.stats["backtracks"],"ms":round(1000*(time.perf_counter()-t),1)})
    c=problem(n,slots); t=time.perf_counter(); sol=csp.min_conflicts(c,random.Random(SEED))
    rows.append({"incidents":n,"slots":slots,"solver":"min-conflicts","solved":bool(sol and csp.verify(c,sol)),"assignments":c.stats["assignments"],"backtracks":0,"ms":round(1000*(time.perf_counter()-t),1)})
csp_df=pd.DataFrame(rows); RESULTS["csp"]=rows; csp_df

,incidents,slots,solver,solved,assignments,backtracks,ms
0,2,3,backtracking,True,96,13,0.2
1,2,3,MRV+LCV,True,72,9,0.9
2,2,3,MRV+FC,True,25,17,0.3
3,2,3,MAC(AC-3),True,8,0,0.7
4,2,3,min-conflicts,True,93,0,2.9
5,3,5,backtracking,True,361,32,0.4
6,3,5,MRV+LCV,True,34,0,2.7
7,3,5,MRV+FC,True,17,5,0.7
8,3,5,MAC(AC-3),True,12,0,3.1
9,3,5,min-conflicts,True,8,0,0.6


## 6. Logic: propositional entailment (resolution vs. model checking)
*Hypothesis H5:* resolution refutation and truth-table model checking agree on every case in
`03_logic/entailment_tests.yaml`.

In [8]:
import yaml
from reasoning.logic import parse_clause, pl_resolution, tt_entails, satisfiable
cases=yaml.safe_load((ROOT/"03_logic/entailment_tests.yaml").read_text())
cnf=[parse_clause(l) for l in (ROOT/"03_logic/cnf_rules.txt").read_text().splitlines() if l.strip() and not l.startswith("#")]
rows=[]
for c in cases["propositional"]:
    kb=cnf+[frozenset([f]) for f in c["facts"]]
    if c["query"] is None:
        rows.append({"id":c["id"],"expect":c["expect"],"resolution":None,"model_check":None,"satisfiable":satisfiable(kb)}); continue
    r,nr=pl_resolution(kb,c["query"]); t,nt=tt_entails(kb,c["query"])
    rows.append({"id":c["id"],"expect":c["expect"],"resolution":r,"resolvents":nr,"model_check":t,"models":nt})
logic_df=pd.DataFrame(rows); RESULTS["propositional"]=rows; logic_df

,id,expect,resolution,resolvents,model_check,models,satisfiable
0,PL1_malware_ioc_is_high_priority,True,True,166.0,True,1024.0,NaN
1,PL2_credential_attack_without_success_not_enta...,False,False,133.0,False,193.0,NaN
2,PL3_containment_from_credential_compromise,True,True,88.0,True,1024.0,NaN
3,PL4_analyst_fp_entails_not_tp,True,True,12.0,True,1024.0,NaN
4,PL5_noncritical_host_not_high_priority,False,False,186.0,False,806.0,NaN
5,PL6_contradiction_is_unsatisfiable,unsatisfiable,None,NaN,None,NaN,False


## 7. First-order logic: the agent's knowledge base and four-way entailment

In [9]:
from agent.knowledge import KnowledgeBase
from agent.entailment import EntailmentChecker
rows=[]
for c in cases["fol"]:
    kb=KnowledgeBase(S,{"assets":{},"users":{},"indicators":[]})
    for f in c["facts"]: kb.tell(f)
    r=EntailmentChecker(S,kb).check(c["query"])
    rows.append({"id":c["id"],"query":c["query"],"expected":c["expect"],"got":r.status.value,"proof_steps":len(r.proof)})
fol_df=pd.DataFrame(rows); RESULTS["fol"]=rows; fol_df

,id,query,expected,got,proof_steps
0,FOL1_entailed_compromise,HighPriorityIncident(srv-db-01),entailed,entailed,13
1,FOL2_unknown_without_evidence,CompromisedHost(srv-db-01),unknown,unknown,0
2,FOL3_contradicted_by_analyst,CompromisedHost(ws-002),contradicted,contradicted,9
3,FOL4_maintenance_scan_refutes,¬TruePositive(A4),entailed,entailed,6
4,FOL5_credential_chain_uncertain_without_success,CompromisedAccount(u-17),uncertain,uncertain,2
5,FOL6_transitive_ontology,"AlertClass(A6, intrusion)",entailed,entailed,7


## 8. Bayesian network: posterior under evidence, explaining away, missing data

In [10]:
from reasoning.bayes import BayesNet
bn=BayesNet.from_json(ROOT/"04_uncertainty/bayes_net.json")
ev_sets={"no evidence":{}, "IOC":{"IOCMatch":True}, "IOC+EDR":{"IOCMatch":True,"EDRDetection":True},
 "AuthAnomaly":{"AuthAnomaly":True,"Maintenance":False}, "AuthAnomaly in maintenance":{"AuthAnomaly":True,"Maintenance":True},
 "IOC + analyst FP":{"IOCMatch":True,"AnalystFP":True}, "Auth anomaly, EDR missing":{"AuthAnomaly":True,"EDRDetection":None,"LateralSignal":False}}
bn_df=pd.DataFrame([{"evidence":k,"P(Compromise)":round(bn.posterior("Compromise",v),3)} for k,v in ev_sets.items()])
RESULTS["bayes"]=bn_df.to_dict(orient="records"); bn_df

,evidence,P(Compromise)
0,no evidence,0.087
1,IOC,0.690
2,IOC+EDR,0.967
3,AuthAnomaly,0.574
4,AuthAnomaly in maintenance,0.177
5,IOC + analyst FP,0.156
6,"Auth anomaly, EDR missing",0.372


In [11]:
# calibration of the BN against the generative model: sample worlds from the BN itself, compare posterior to frequency
rng=np.random.default_rng(SEED); samples=[]
for _ in range(20000):
    a={}
    for n in bn.nodes: a[n.name]=rng.random()<n.p_true(a)
    samples.append(a)
ev={"IOCMatch":True,"AuthAnomaly":True}
match=[s for s in samples if all(s[k]==v for k,v in ev.items())]
emp=np.mean([s["Compromise"] for s in match]); RESULTS["bayes_check"]={"exact":bn.posterior("Compromise",ev),"sampled":float(emp),"n":len(match)}
RESULTS["bayes_check"]

{'exact': 0.9609628869505731, 'sampled': 0.9532110091743119, 'n': 1090}

## 9. Temporal reasoning: HMM filtering, Viterbi decoding, and Kalman burst detection
*Hypothesis H6:* Viterbi recovers the hidden kill-chain stage far better than a "most common observation" baseline,
and the Kalman innovation flags bursts with few false alarms.

In [12]:
from reasoning.temporal import HMM, kalman_rate
hmm=HMM.from_json(ROOT/"04_uncertainty/hmm.json"); rng=np.random.default_rng(SEED)
obs_to_state={"quiet":"benign","scan":"recon","auth_anomaly":"initial_access","malware":"initial_access","lateral":"lateral_movement","exfil":"exfiltration"}
acc_v,acc_b,acc_f=[],[],[]
for _ in range(300):
    xs,os_=hmm.sample(30,rng)
    acc_v.append(np.mean([a==b for a,b in zip(xs,hmm.viterbi(os_))]))
    acc_b.append(np.mean([a==obs_to_state[o] for a,o in zip(xs,os_)]))
    acc_f.append(np.mean([xs[t]==hmm.states[int(hmm.filter(os_[:t+1]).argmax())] for t in range(len(os_))]))
RESULTS["hmm"]={"viterbi_acc":float(np.mean(acc_v)),"filter_acc":float(np.mean(acc_f)),"naive_baseline_acc":float(np.mean(acc_b))}
RESULTS["hmm"]

{'viterbi_acc': 0.773888888888889,
 'filter_acc': 0.7326666666666667,
 'naive_baseline_acc': 0.6302222222222222}

In [13]:
fa=det=0; trials=500
for _ in range(trials):
    base=rng.poisson(0.4,20).astype(float)
    if kalman_rate(list(base)).max_z>S.thresholds.kalman_anomaly_sigma: fa+=1
    b=base.copy(); b[15]+=6
    if kalman_rate(list(b)).max_z>S.thresholds.kalman_anomaly_sigma: det+=1
RESULTS["kalman"]={"false_alarm_rate":fa/trials,"detection_rate":det/trials}
RESULTS["kalman"]

{'false_alarm_rate': 0.016, 'detection_rate': 1.0}

## 10. Machine learning, neural network and RL (from `05_ml/train.py`)
The training pipeline is a script so that CI can reproduce it. Here we load its outputs.

In [14]:
M=json.loads((ROOT/"05_ml/artifacts/metrics.json").read_text())
ml_df=pd.DataFrame(M["test"]).T[["roc_auc","f1","precision","recall","fpr","ece"]].round(3)
RESULTS["ml_selected"]=M["selected_model"]; RESULTS["learning_case"]=M["learning_case"]
print("selected:",M["selected_model"]); ml_df

selected: mlp


,roc_auc,f1,precision,recall,fpr,ece
majority,0.500,0.000,0.000,0.000,0.000,0.000
symbolic_rules,0.798,0.518,0.926,0.359,0.013,0.051
sklearn_random_forest,0.956,0.841,0.853,0.829,0.063,0.034
logistic_regression,0.960,0.846,0.864,0.829,0.058,0.027
mlp,0.962,0.846,0.864,0.829,0.058,0.023


In [15]:
pd.DataFrame({"q_learning":M["rl"]["q_learning"],"rule_policy":M["rl"]["rule_policy"]}).round(3)

,q_learning,rule_policy
avg_reward,3.548,1.484
fp_containment_rate,0.000,0.000
missed_compromise_rate,0.000,0.035
unsafe_action_rate,0.000,0.000


## 11. End-to-end: the six demonstration scenarios

In [16]:
from agent.orchestrator import Agent
from sim.scenarios import load_scenario
import tempfile
s2=load_settings(overrides={"paths.runtime_dir":tempfile.mkdtemp()})
agent=Agent(s2); rows=[]
for name in ["normal","uncertain","contradiction","constraint","adversarial"]:
    sc=load_scenario(name); r=agent.run_case(sc["observations"],scenario=name,action_requests=sc.get("action_requests"))
    top=r.inference.risks[0] if r.inference.risks else None
    rows.append({"scenario":name,"iterations":r.iterations,"top_asset":top.asset_id if top else None,
                 "P(compromise)":round(top.p_compromise_bayes,2) if top else None,
                 "entailed":sum(e.status.value=="entailed" for e in r.entailments),
                 "contradicted":sum(e.status.value=="contradicted" for e in r.entailments),
                 "pending_approval":[f"{p.action.value}({p.target})" for p in r.proposals if p.status.value=="pending_approval"],
                 "blocked":[f"{p.tool}({p.target})" for p in r.proposals if p.status.value=="blocked"]})
e2e=pd.DataFrame(rows); RESULTS["scenarios"]=rows; e2e

,scenario,iterations,top_asset,P(compromise),entailed,contradicted,pending_approval,blocked
0,normal,2,ws-005,0.94,4,1,[isolate_host(ws-005)],[]
1,uncertain,2,ws-013,0.43,0,0,[],[]
2,contradiction,1,ws-008,0.58,0,3,[],[isolate_host(ws-008)]
3,constraint,2,dc-01,0.99,10,0,"[isolate_host(jump-01), disable_account(adm-01...","[isolate_host(dc-01), disable_account(svc-back..."
4,adversarial,2,ws-011,0.01,0,0,[],"[delete_audit_log(audit), isolate_host(dc-01),..."


## 12. Hypotheses, results and failure cases
| # | Hypothesis | Result |
|---|---|---|
| H1 | A* is cost-optimal and expands fewer nodes than UCS | See §2 table (optimal_rate = 1.0 for UCS/A*) |
| H2 | Local search reaches the exhaustive optimum | Yes after raising restarts (failure case documented in §3) |
| H3 | Alpha-beta = minimax value with fewer nodes | See §4 node counts |
| H4 | MRV/LCV/FC/MAC beat plain backtracking | See §5. MAC needs the fewest assignments and proves the over-constrained instance unsatisfiable fastest. MRV+LCV without propagation can be *worse* (4 incidents), which is a useful counter-example |
| H5 | Resolution ≡ model checking | See §6 |
| H6 | Viterbi ≫ naive stage guess; Kalman low false-alarm | See §9 |

**Failure cases recorded**
1. Hill-climbing plateau (§3), fixed with more restarts.
2. `ipaddress.is_private` treats the RFC 5737 documentation ranges as private, so `BlockWarranted` was never derived.
   Fixed by an explicit list of internal networks (`agent/knowledge.py`).
3. Rank-based ROC-AUC without tie handling scored a constant predictor 1.0. Fixed by averaging ranks.
4. Alpha-beta move ordering broke ties between equally critical hosts by set iteration order, which changes with
   `PYTHONHASHSEED`. The segmentation point flipped between `jump-01` and `srv-app-01` from run to run. Fixed by
   breaking ties on the host name.
5. The HMM labels a single auth anomaly as "benign" stage. This is expected with a strong benign prior, so stage
   evidence is only one input to fusion and never a trigger.

In [17]:
out=ROOT/"data/results/simulation_metrics.json"; out.parent.mkdir(parents=True,exist_ok=True)
out.write_text(json.dumps(RESULTS,indent=1,default=str)); print("wrote",out.relative_to(ROOT))

wrote data/results/simulation_metrics.json
